# GLASS Interactive SI Shift

Display interactive latest-observed-count vs future SI-shift plots, with GLASS pathogen-antibiotic pairs highlighted.

In [10]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
from pathlib import Path
import sys

from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.pipelines import (
    build_glass_match_table,
    build_novel_prediction_table,
    build_si_analysis_paths,
    load_or_build_interactive_rank,
    load_si_and_clean_glass_list,
    plot_future_si_vs_si_shift,
)
from amr.si import plot_interactive_latest_observed_count_vs_future_si_shift

## Parameters

In [12]:
OUTPUT_DIR = PROJECT_ROOT / "outputs"
PARQUET_OUTPUT_DIR = OUTPUT_DIR / "parquets"
PARQUET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = PARQUET_OUTPUT_DIR
ASSETS_DIR = PROJECT_ROOT / "assets"

ANTIBIOTIC_ANALYSIS_LEVEL = "High level categories"  # antibiotic, Class, Simplified categories, High level categories, Origin
SI_PATHS = build_si_analysis_paths(PROJECT_ROOT, ANTIBIOTIC_ANALYSIS_LEVEL)
YEARLY_SI_PATH = SI_PATHS["yearly_si"]
RANK_OUTPUT_PATH = SI_PATHS["rank_output"]

GLASS_XLSX_PATH = ASSETS_DIR / "GLASS list.xlsx"
GLASS_CLEANED_PATH = ASSETS_DIR / "glass_amr_cleaned.csv"
BACTERIA_SYNONYM_PATH = ASSETS_DIR / "bacteria_synonym_lookup.csv"
ANTIBIOTIC_SYNONYM_PATH = ASSETS_DIR / "antibiotic_synonym_lookup.csv"
ANTIBIOTIC_COARSENING_PATH = ASSETS_DIR / "antibiotic_coarsening_lookup.csv"

FIT_START_YEAR = 2000
FIT_END_YEAR = 2023
NUM_FUTURE_YEARS = 2
LOESS_FRAC = 0.6
LOESS_DEGREE = 1
TAIL_DAMPING_YEARS = None

MIN_LAST_AVAILABLE_COUNT = 1
MIN_COUNT_ABS_PCT_ERROR_MEAN = 0
MAX_COUNT_ABS_PCT_ERROR_MEAN = 100
MIN_COUNTS_BACKGROUND = 10
MIN_ABS_DELTA_SI = 0.01
MIN_SI_DELTA = None

DISPLAY_NAME = True
MAX_TEXT_LABELS = 30
WIDTH = 1400
HEIGHT = 900

## Load SI And Clean GLASS List

In [13]:
si_df, glass_pairs_df = load_si_and_clean_glass_list(
    YEARLY_SI_PATH,
    GLASS_XLSX_PATH,
    bacteria_synonym_path=BACTERIA_SYNONYM_PATH,
    antibiotic_synonym_path=ANTIBIOTIC_SYNONYM_PATH,
    antibiotic_coarsening_path=ANTIBIOTIC_COARSENING_PATH,
    glass_cleaned_path=GLASS_CLEANED_PATH,
)
display(glass_pairs_df.head(20))

Loaded SI rows: 107 from /media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/outputs/parquets/yearly_si_high_level_categories.parquet
Pathogens: 5; antibiotics/levels: 2
Saved cleaned GLASS rows: 89 to /media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/assets/glass_amr_cleaned.csv


,raw_pathogen,raw_antibiotic,pathogen,antibiotic,pathogen_standardized,antibiotic_standardized,Class,Simplified categories,High level categories,Origin
0,Acinetobacter spp.,amikacin,Acinetobacter spp.,Amikacin,False,True,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
1,Acinetobacter spp.,Colistin,Acinetobacter spp.,Colistin,False,False,Polymyxins,Miscellaneous,Other,Naturally occurring/modified
2,Acinetobacter spp.,doripenem,Acinetobacter spp.,Doripenem,False,True,Carbapenems,Penems/Carbapenems,Beta lactam and related,Naturally occurring/modified
3,Acinetobacter spp.,Gentamicin,Acinetobacter spp.,Gentamicin,False,False,Aminoglycosides,Aminocyclitols/Aminoglycosides,Aminocyclitols/Aminoglycosides,Naturally occurring/modified
4,Acinetobacter spp.,Imipenem,Acinetobacter spp.,Imipenem/cilastatin,False,True,Carbapenems,Penems/Carbapenems,Beta lactam and related,Naturally occurring/modified
5,Acinetobacter spp.,meropenem,Acinetobacter spp.,Meropenem,False,True,Carbapenems,Penems/Carbapenems,Beta lactam and related,Naturally occurring/modified
6,Acinetobacter spp.,minocycline,Acinetobacter spp.,Minocycline,False,True,Tetracyclines,Tetracyclines and related,Tetracyclines and related,Naturally occurring/modified
7,Acinetobacter spp.,Tigecycline,Acinetobacter spp.,Tigecycline,False,False,Glycylcyclines,Tetracyclines and related,Tetracyclines and related,Naturally occurring/modified
8,E. coli,Cefepime,Escherichia coli,Cefepime,True,False,Fourth-generation-cephalosporins,Cephalosporins,Beta lactam and related,Naturally occurring/modified
9,E. coli,cefotaxime,Escherichia coli,Cefotaxime,True,True,Third-generation-cephalosporins,Cephalosporins,Beta lactam and related,Naturally occurring/modified


## Build Or Load Rank Data

In [14]:
rank_df = load_or_build_interactive_rank(
    si_df,
    RANK_OUTPUT_PATH,
    fit_start_year=FIT_START_YEAR,
    fit_end_year=FIT_END_YEAR,
    num_future_years=NUM_FUTURE_YEARS,
    loess_frac=LOESS_FRAC,
    loess_degree=LOESS_DEGREE,
    tail_damping_years=TAIL_DAMPING_YEARS,
    verbose=True,
)
display(rank_df.head())

Loaded cached rank rows: 9 from /media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/Antimicrobial-Resistance/outputs/parquets/yearly_si_ranked_future_high_level_categories.parquet


,pathogen,antibiotic,reference_year,future_year,num_future_years,fit_start_year,fit_end_year,latest_available_year,n_pair_years,current_si_source,...,post_fit_si_abs_pct_error_mean,loess_frac,loess_degree,tail_damping_years,status,error,current_rank,future_rank,rank_change,abs_rank_change
0,Haemophilus influenzae,Aminocyclitols/Aminoglycosides,2023,2025,2,2010,2023,2019,4,predicted,...,NaN,0.6,1,None,ok,None,7.0,6.0,1.0,1.0
1,Mycobacterium tuberculosis,Aminocyclitols/Aminoglycosides,2023,2025,2,2010,2023,2024,8,predicted,...,145.722583,0.6,1,None,ok,None,6.0,7.0,-1.0,1.0
2,Clostridioides difficile,Beta lactam and related,2023,2025,2,2010,2023,2024,9,predicted,...,236.133764,0.6,1,None,ok,None,1.0,1.0,0.0,0.0
3,Salmonella enterica,Beta lactam and related,2023,2025,2,2010,2023,2026,17,observed,...,73.950193,0.6,1,None,ok,None,2.0,2.0,0.0,0.0
4,Klebsiella pneumoniae,Aminocyclitols/Aminoglycosides,2023,2025,2,2010,2023,2026,17,observed,...,768.282827,0.6,1,None,ok,None,3.0,3.0,0.0,0.0


## Interactive Latest Count Vs Future SI Shift

In [15]:
latest_count_shift_result = plot_interactive_latest_observed_count_vs_future_si_shift(
    rank_df,
    glass_pairs_df=glass_pairs_df,
    min_last_available_count=MIN_LAST_AVAILABLE_COUNT,
    min_count_abs_pct_error_mean=MIN_COUNT_ABS_PCT_ERROR_MEAN,
    max_count_abs_pct_error_mean=MAX_COUNT_ABS_PCT_ERROR_MEAN,
    min_counts_background=MIN_COUNTS_BACKGROUND,
    min_abs_delta_si=MIN_ABS_DELTA_SI,
    min_si_delta=MIN_SI_DELTA,
    display_name=DISPLAY_NAME,
    max_text_labels=MAX_TEXT_LABELS,
    width=WIDTH,
    height=HEIGHT,
    show=True,
)
latest_count_shift_df = latest_count_shift_result["plot_df"]
print(f"Displayed rows: {len(latest_count_shift_df):,}; GLASS matches: {int(latest_count_shift_df['is_glass_pair'].sum()):,}")

Displayed rows: 6; GLASS matches: 2


## Display GLASS Matches In Plot

In [16]:
glass_plot_matches_df = build_glass_match_table(latest_count_shift_df)
display(glass_plot_matches_df)

,pathogen,antibiotic,latest_available_year,future_year,latest_available_count,future_predicted_count,latest_smoothed_SI,future_predicted_SI,future_minus_latest_smoothed_SI,post_fit_count_abs_pct_error_mean,glass_group_column
0,Klebsiella pneumoniae,Beta lactam and related,2026,2025,41.0,121.537779,0.640590,0.166719,-0.473871,82.949172,High level categories
1,Salmonella enterica,Beta lactam and related,2026,2025,22.0,70.299392,0.811303,0.372485,-0.438817,71.320219,High level categories


## Novel Non-GLASS Predictions

These are plotted pairs that do not appear in the cleaned GLASS list, sorted as candidate novel predictions of interest.

In [17]:
novel_prediction_df = build_novel_prediction_table(latest_count_shift_df)
print(f"Novel non-GLASS predictions displayed: {len(novel_prediction_df):,}")
display(novel_prediction_df)

Novel non-GLASS predictions displayed: 4


,pathogen,antibiotic,latest_available_year,future_year,latest_available_count,future_predicted_count,latest_smoothed_SI,future_predicted_SI,future_minus_latest_smoothed_SI,post_fit_count_abs_pct_error_mean
0,Mycobacterium tuberculosis,Beta lactam and related,2024,2025,1.0,1.000000,-0.453348,-0.453348,0.000000,0.000000
1,Mycobacterium tuberculosis,Aminocyclitols/Aminoglycosides,2024,2025,4.0,0.163764,-0.181089,-0.237044,-0.055954,91.905222
2,Salmonella enterica,Aminocyclitols/Aminoglycosides,2026,2025,4.0,3.288578,-0.002462,-0.379382,-0.376920,48.125325
3,Klebsiella pneumoniae,Aminocyclitols/Aminoglycosides,2026,2025,20.0,57.868483,0.727917,0.317237,-0.410680,90.784692


## Future SI Vs Future SI Shift

This uses the same y-axis definition as the latest-count plot: predicted future SI minus smoothed SI at the latest available year.

In [18]:
future_si_shift_result = plot_future_si_vs_si_shift(
    latest_count_shift_df,
    min_count_abs_pct_error_mean=MIN_COUNT_ABS_PCT_ERROR_MEAN,
    max_count_abs_pct_error_mean=MAX_COUNT_ABS_PCT_ERROR_MEAN,
    display_name=DISPLAY_NAME,
    max_text_labels=MAX_TEXT_LABELS,
    width=WIDTH,
    height=HEIGHT,
    show=True,
)
future_si_shift_fig = future_si_shift_result["figure"]